In [ ]:
#Final Seminar
!pip install -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu gradio google-genai scikit-learn

In [2]:
from google.colab import drive
import os
import pickle

try:
    drive.mount('/content/drive')
    print("Drive mounted")
except Exception as e:
    print(f"Drive mount failed: {e}")

NOVEL_DIR = "/content/drive/MyDrive/novels"
SAVE_DIR = "/content/drive/MyDrive/novels/processed_data"
os.makedirs(SAVE_DIR, exist_ok=True)

NOVEL_FILES = {
    "အမှတ်တရ (ဂျူး)": "AMhatTaYa-Juu.txt",
    "ပြုံး၍လဲကန်တော့ခံတော်မူပါ ရယ်၍လဲကန်တော့ခံတော်မူပါ (နုနုရည်အင်းဝ)": "NuNuYiInnWa.txt",
    "ဗိုလ်ချုပ်အောင်ဆန်း အတ္ထုပ္ပတ္တိ (သခင်ကိုယ်တော်မှိုင်း)": "General_Aung_San_Biography.txt",
    "သူလိုလူ (ဂျာနယ်ကျော်မမလေး)": "Thu_Lo_Lu.txt",
    "လက်ကျန်လရောင် (ပုညခင်)": "LattKyanLaYaung.txt",
}
print("Novels config loaded")

Mounted at /content/drive
Drive mounted
Novels config loaded


In [3]:
import re
def clean_text(text):
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.replace("\t", " ")
    return text.strip()

In [4]:
def chunk_text_sentence(text, max_chars=500, overlap=1):
    """
    Sentence-aware chunking for Burmese text with sentence overlap.
    Feature: Prevents context loss between chunks.
    """
    if not text: return []
    sentences = re.split(r"(?<=[။!?])\s*", text)
    chunks, current = [], []

    for sent in sentences:
        sent = sent.strip()
        if not sent: continue

        if sum(len(s) for s in current) + len(sent) <= max_chars:
            current.append(sent)
        else:
            if current:
                chunks.append(" ".join(current))
                current = current[-overlap:] if overlap > 0 else []

            if len(sent) > max_chars:
                chunks.append(sent)
                current = []
            else:
                current.append(sent)

    if current:
        chunks.append(" ".join(current))
    return chunks

In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

LABSE_CACHE_DIR = "/content/drive/MyDrive/models/labse"
os.makedirs(LABSE_CACHE_DIR, exist_ok=True)

print("⏳ Loading LaBSE embeddings...")
try:
    embed_model = SentenceTransformer("sentence-transformers/LaBSE", cache_folder=LABSE_CACHE_DIR)
except Exception as e:
    print(f"Model Load Error: {e}")

novels = {}

for novel_name, filename in NOVEL_FILES.items():
    save_path = os.path.join(SAVE_DIR, f"{filename}.pkl")
    index_path = os.path.join(SAVE_DIR, f"{filename}.index")

    try:
        # If both files exist, just load
        if os.path.exists(save_path) and os.path.exists(index_path):
            print(f"Loading persistent index: {novel_name}")
            index = faiss.read_index(index_path)
            with open(save_path, "rb") as f:
                chunks = pickle.load(f)

        #  Else, create new chunks & IVF index and save them
        else:
            print(f"Creating new index for: {novel_name}...")
            path = os.path.join(NOVEL_DIR, filename)
            with open(path, "r", encoding="utf-8") as f:
                raw_text = f.read()

            text = clean_text(raw_text)
            chunks = chunk_text_sentence(text)
            embeddings = embed_model.encode(chunks, show_progress_bar=True).astype("float32")

            dim = embeddings.shape[1]

            # Advanced IVF: dynamic clusters
            nlist = max(10, min(100, len(embeddings)//5))
            quantizer = faiss.IndexFlatL2(dim)
            index = faiss.IndexIVFFlat(quantizer, dim, nlist)

            # Train and add embeddings
            index.train(embeddings)
            index.add(embeddings)

            # nprobe tuning for speed vs accuracy
            index.nprobe = min(10, nlist)

            # Save index & chunks to processed_data
            faiss.write_index(index, index_path)
            with open(save_path, "wb") as f:
                pickle.dump(chunks, f)

        novels[novel_name] = {"chunks": chunks, "index": index}

    except Exception as e:
        print(f"Skip {novel_name}: {e}")

print("🎉 All novels ready and indexes created in processed_data")

print("\nTotal Chunks Per Novel:\n")

for novel_name, data in novels.items():
    total_chunks = len(data["chunks"])
    print(f"{novel_name} → {total_chunks} chunks")

total_all = sum(len(data["chunks"]) for data in novels.values())
print(f"\nTotal Chunks (All Novels): {total_all}")


⏳ Loading LaBSE embeddings...


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading persistent index: အမှတ်တရ (ဂျူး)
Loading persistent index: ပြုံး၍လဲကန်တော့ခံတော်မူပါ ရယ်၍လဲကန်တော့ခံတော်မူပါ (နုနုရည်အင်းဝ)
Loading persistent index: ဗိုလ်ချုပ်အောင်ဆန်း အတ္ထုပ္ပတ္တိ (သခင်ကိုယ်တော်မှိုင်း)
Loading persistent index: သူလိုလူ (ဂျာနယ်ကျော်မမလေး)
Loading persistent index: လက်ကျန်လရောင် (ပုညခင်)
🎉 All novels ready and indexes created in processed_data

Total Chunks Per Novel:

အမှတ်တရ (ဂျူး) → 428 chunks
ပြုံး၍လဲကန်တော့ခံတော်မူပါ ရယ်၍လဲကန်တော့ခံတော်မူပါ (နုနုရည်အင်းဝ) → 663 chunks
ဗိုလ်ချုပ်အောင်ဆန်း အတ္ထုပ္ပတ္တိ (သခင်ကိုယ်တော်မှိုင်း) → 1659 chunks
သူလိုလူ (ဂျာနယ်ကျော်မမလေး) → 2680 chunks
လက်ကျန်လရောင် (ပုညခင်) → 1339 chunks

Total Chunks (All Novels): 6769


In [6]:
from google import genai
from google.colab import userdata

try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=API_KEY)
    print("Gemma 3 API Client Ready")
except Exception as e:
    print(f"API Setup Error: {e}. Check Colab Secrets.")

Gemma 3 API Client Ready


In [7]:
def build_prompt(context, topic):

    return f"""
[ROLE]
You are a distinguished Burmese Literary Critic writing for an academic literary journal.
Your writing must feel natural, intelligent, and human-like — never robotic.

[SOURCE DATA – THE ONLY PERMITTED KNOWLEDGE]
{context}

[CORE OPERATING PRINCIPLES]

1. STRICT SEMANTIC BOUNDARY:
   Operate in absolute STRICT MODE.
   Only write statements that are clearly supported by the source data above.
   Do NOT assume, invent, expand, or recall from outside knowledge.
   Do NOT add background information that is not explicitly present.

2. ADAPTIVE RESPONSE INTELLIGENCE:
   Automatically adjust the depth and length of your response based on the topic:

   • If the topic asks for a summary, overview, or full story:
     Reconstruct the storyline smoothly and logically.
     Present events in coherent narrative order.
     Ensure natural flow from beginning to development to conclusion (if present).
     Avoid fragmented or disconnected transitions.

   • If the topic asks for theme, meaning, symbolism, or interpretation:
     Provide deep literary analysis with intellectual clarity.

   • If the topic asks about a character:
     Explain personality, motivations, development, and narrative role.

   • If the topic asks for comparison:
     Clearly explain similarities and differences in a balanced manner.

   • If the topic asks a short factual question (who, when, where):
     Answer concisely and precisely without unnecessary expansion.

3. NARRATIVE COHERENCE:
   Ensure logical continuity.
   Avoid jumping randomly between unrelated events.
   Merge all relevant ideas into ONE smooth and cohesive paragraph.

4. IMMERSION RULE:
   Do NOT mention:
   - source data
   - fragments
   - sections
   - chunks
   - provided text
   Speak naturally as if discussing the novel itself.

5. LANGUAGE POLICY (STRICT ENFORCEMENT):
   - Use ONLY Burmese and English.
   - Do NOT generate Thai, Hindi, Sanskrit, Korean , Pali, Dhivehi or any other foreign scripts.
   - Do NOT mix unrelated foreign characters.
   - If unusual foreign words appear, rewrite them naturally in Burmese or English.

6. STYLE REQUIREMENTS:
   - Formal literary Burmese when writing Burmese.
   - Clear academic English when writing English.
   - Human-like, smooth, and intellectually refined.
   - No bullet points.
   - No headings.
   - No introductory phrases such as:
     "Certainly..."
     "Here is the explanation..."
   Begin immediately with meaningful content.
   End naturally without meta-commentary.

7. TRUTH PROTOCOL:
   If the requested topic is not supported by the source data,
   output ONLY:
   ဝတ္ထုအတွင်း မတွေ့ပါ။

[ANALYSIS TOPIC]
{topic}

[WRITE THE RESPONSE BELOW]
"""


In [8]:
import re

def clean_generated_text(text):
    text = re.sub(r'[^A-Za-z0-9\u1000-\u109F\s\.,;:!?()\-"“”‘’—…%]', '', text)
    sentences = re.split(r'(?<=[။!?])', text)
    seen = set()
    clean_sentences = []

    for s in sentences:
        s = s.strip()
        if not s or s in seen:
            continue
        seen.add(s)
        clean_sentences.append(s)
    if clean_sentences and not re.search(r'[။!?]$', clean_sentences[-1]):
        clean_sentences.pop()

    return " ".join(clean_sentences)


In [9]:
def generate_text(context, topic):
    prompt = build_prompt(context, topic)
    try:
        response = client.models.generate_content(
            model="gemma-3-27b-it",
            config=genai.types.GenerateContentConfig(temperature=0.35, max_output_tokens=800),
            contents=prompt
        )
        text = response.text
        if "Analysis:" in text: text = text.split("Analysis:")[-1]
        return text.strip()
    except Exception as e:
        return f"Error connecting to Gemma 3: {str(e)}"

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def rerank_chunks(query, retrieved_chunks):
    if not retrieved_chunks: return []
    query_emb = embed_model.encode([query])
    chunk_embs = embed_model.encode(retrieved_chunks)
    similarities = cosine_similarity(query_emb, chunk_embs)[0]
    scored = sorted(zip(similarities, retrieved_chunks), key=lambda x: x[0], reverse=True)
    return [c[1] for c in scored[:10]]

In [11]:
def describe_topic(topic, novel_name, top_k=10):
    """
    Retrieve relevant chunks using IVF + rerank for accuracy,
    then generate Burmese literary explanation.

    Args:
        topic (str): Query/topic in Burmese.
        novel_name (str): Name of the novel in NOVEL_FILES.
        top_k (int): Number of top chunks to consider.

    Returns:
        tuple: (Generated explanation, joined_top_chunks)
    """

    if not topic.strip():
        return "ခေါင်းစဉ် ရေးပါ။", ""

    novel = novels.get(novel_name)
    if not novel:
        return "ဝတ္ထုဒေတာ မရှိပါ။", ""

    try:
        # Encode topic into embedding
        q_emb = embed_model.encode([topic]).astype("float32")

        # IVF search in index (high-speed cluster search)
        D, I = novel["index"].search(q_emb, top_k * 2)  # Broad search
        raw_retrieved = [novel["chunks"][i] for i in I[0] if i != -1]

        if not raw_retrieved:
            return "အချက်အလက် မတွေ့ပါ။", ""

        # Rerank retrieved chunks by cosine similarity
        best_chunks = rerank_chunks(topic, raw_retrieved)[:top_k]

        # Join top 10 chunks for UI display
        joined_chunks = "\n\n━━━━━━━━━━━━━━\n\n".join(best_chunks)

        # Prepare context for LLM
        context = "\n\n".join(
            [f"[chunk {i+1}]: {c}" for i, c in enumerate(best_chunks)]
        )

        # Generate explanation
        explanation = generate_text(context, topic)

        final_answer = (
            f"💗 **{novel_name} ဝတ္ထုအပေါ် အခြေခံထားသော ဖော်ပြချက်**\n\n"
            f"{clean_generated_text(explanation)}\n\n"
            "━━━━━━━━━━━━━━\n"
            "ဤဖော်ပြချက်သည် ဝတ္ထုအတွင်းပါ အချက်အလက်များကိုသာ အခြေခံထားပါသည်။"
        )

        return final_answer, joined_chunks

    except Exception as e:
        return f"❗ System Error: {str(e)}", ""


In [12]:
import gradio as gr
import time

MAX_CHARS = 450

def user_interact(topic, novel_name, history):
    topic = topic.strip()
    textbox_clear = ""

    if not topic:
        yield history, textbox_clear
        return

    # Input too long → warning in textbox
    if len(topic) > MAX_CHARS:
        yield history, f"⚠️ Too long! Limit {MAX_CHARS} chars (your input: {len(topic)})"
        return

    # Valid input → add user message
    history.append({"role": "user", "content": f"🌸 {topic}"})
    yield history, textbox_clear

    # Show thinking message
    history.append({"role": "assistant", "content": "💌 Bot is thinking..."})
    yield history, textbox_clear

    # Generate bot response
    start = time.time()
    response, joined_chunks = describe_topic(topic, novel_name)
    duration = time.time() - start

    # hidden chunks
    history[-1]["content"] = (
        f"{response}\n\n"
        "<details>"
        "<summary>🔍 Show Top 10 Retrieved Chunks</summary>\n\n"
        f"{joined_chunks}"
        "\n</details>\n\n"
        f"<sub>Bot response time: {duration:.2f} sec</sub>"
    )

    yield history, textbox_clear


# Gradio UI
with gr.Blocks() as demo:

    novel_selector = gr.Dropdown(
        choices=list(NOVEL_FILES.keys()),
        value=list(NOVEL_FILES.keys())[0],
        label="📚💕 ဝတ္ထု ရွေးချယ်ပါ"
    )

    chatbot = gr.Chatbot(height=400, elem_id="chatbot")
    state = gr.State([])

    with gr.Row():
        msg = gr.Textbox(
            placeholder="ဝတ္ထုနှင့် ပတ်သက်သော အကြောင်းအရာ မေးပါ....",
            scale=4,
            lines=1
        )
        send = gr.Button("Send 💌", scale=1)

    # Connect function to button and Enter key
    send.click(
        user_interact,
        inputs=[msg, novel_selector, state],
        outputs=[chatbot, msg],
        queue=True
    )

    msg.submit(
        user_interact,
        inputs=[msg, novel_selector, state],
        outputs=[chatbot, msg],
        queue=True
    )

    demo.css = """
    body {background:#fff0f6; font-family:'Segoe UI', sans-serif;}
    #chatbot {border-radius:20px; border:2px solid #ff85c1; padding:10px;}
    button {
        background:linear-gradient(120deg,#ff69b4,#ff85c1);
        color:white;
        font-weight:bold;
        border-radius:15px;
    }
    .gr-chatbot-message.user {
        background:#ffe4f0; border-radius:15px; text-align:right;
    }
    .gr-chatbot-message.assistant {
        background:#ff69b4; color:white; border-radius:15px;
    }
    """

demo.launch(share=True, theme=gr.themes.Base())


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c56ef28bb434a2abc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
